## **Spectroscopic IROS Reconstruction**

In [ ]:
from pathlib import Path

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
import darksun as ds

ds.show.set_figures_darkbkg()

In [ ]:
BASE_PATH: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data"
SIM_PATH: str = f"{BASE_PATH}/Simulations"
OUT_PATH: str = f"{BASE_PATH}/Outputs"

# BASE_PATH: str = "/mnt/d/PhD_AASS/Coding/Images_fits"
# SIM_PATH = OUT_PATH = BASE_PATH

In [ ]:
# MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
MASK_FITS: str = "mask_NTHT_20250725.fits"
UPS_X: int = 2
UPS_Y: int = 1

wfm: CodedMaskCamera = codedmask(f"{SIM_PATH}/{MASK_FITS}", UPS_X, UPS_Y)

In [ ]:
# SKYFIELD: str = "IROSDummy"
# DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_1ks"
SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks"

ID_CAMERA_A: str = "cam1a"
DATASET: str = "reconstructed"

filepaths: dict[str, dict[str, Path]] = simulation_files(f"{SIM_PATH}/{SKYFIELD}/{DATA_FITS}")

E_min, E_max = 2.0, 6.0   # [keV]

sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max)

In [ ]:
from typing import Any

import numpy as np
from numpy.typing import NDArray

from darksun.data import Log

from IROSrec.iros.optim import iros_singleCAM
from IROSrec.iros.procedure import run_IROS, get_sources_database

In [ ]:
E_min, E_max = 2.0, 6.0   # [keV]

sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max)

In [ ]:
def eband_schedule(emin: float, emax: float, ebin: float) -> dict[int, tuple[float, float]]:
    """Defines a schedule for the energy bands to analyse."""
    if not (emax > emin) or (emax - emin < ebin):
        raise ValueError('Invalid energy band bound values.')
     
    nruns = int((emax - emin) / ebin)
    schedule = {run: (float(emin), float(emax - run * ebin)) for run in range(nruns)}
    return schedule

In [ ]:
def select_target_srcs() -> ...:
    """Filters the input photon list for the given sources equatorial coords."""
    raise NotImplementedError


def select_eband_phs() -> ...:
    """Filters the input photon list in the given energy band."""
    raise NotImplementedError

In [ ]:
def run(
    camera: CodedMaskCamera,
    detector: NDArray,
    max_iterations: int,
    camID: str | None = None,
    **iros_kwargs: Any,
) -> Log:
    """Runs the IROS procedure and stores optimised sources params."""
    loop = iros_singleCAM(camera, detector, max_iterations, **iros_kwargs)
    log, _ = run_IROS(camera, loop, camID)
    return log